In [10]:
import sys
sys.path.insert(0, "../src")
import importlib
importlib.reload(o2)

from pyspark.sql import SparkSession, functions as F
import main, m1, o2

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("dev-o2")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

In [11]:
trips = main.load_trips(
    spark, "../databases", [2025], 1.0
)

trips_clean = m1.clean(
    m1.flag_invalid(trips, years=[2025])
)

zones = main.load_zones(
    spark, "../databases/taxi_zone_lookup.csv"
)

print("Clean trips:", trips_clean.count())

Clean trips: 8482202


In [18]:
print("Zones columns:", zones.columns)
zones.printSchema()

print("Flows columns:", flows.columns)
print("Pickup columns:", pickup.columns)
print("Dropoff columns:", dropoff.columns)

Zones columns: ['LocationID', 'Borough', 'Zone', 'service_zone']
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



NameError: name 'flows' is not defined

In [15]:
import importlib
importlib.reload(o2)
flows = o2.build_monthly_flows(trips_clean, zones)

flows.printSchema()
flows.show(10, truncate=False)

AnalysisException: [UNRESOLVED_USING_COLUMN_FOR_JOIN] USING column `DOLocationID` cannot be resolved on the left side of the join. The left-side columns: [`PULocationID`, `month`, `outbound_trips`, `pickup_borough`, `pickup_zone`, `route_share`, `trips`, `year`]. SQLSTATE: 42703

In [9]:
share_check = (
    flows
    .groupBy("year", "month", "PULocationID")
    .agg(
        F.sum("route_share").alias("sum_route_shares")
    )
)

share_check.select(
    F.min("sum_route_shares").alias("min_share_sum"),
    F.max("sum_route_shares").alias("max_share_sum")
).show()

NameError: name 'flows' is not defined

In [ ]:
top20 = o2.top_routes(flows, n=20)

top20.orderBy(
    "year",
    "month",
    "pickup_borough",
    "route_rank"
).show(50, truncate=False)

In [ ]:
reversals = o2.share_reversals(flows)

reversals.orderBy(
    "year",
    "pickup_borough",
    "PULocationID",
    "DOLocationID"
).show(50, truncate=False)

In [ ]:
print("Monthly route rows:", flows.count())
print("Top-20 rows:", top20.count())
print("Share reversals:", reversals.count())

o2.write(
    flows,
    top20,
    reversals,
    "../output"
)

In [ ]:
flows.explain(mode="formatted")